# Задача 1 - SFT перенос стиля

обучаем Qwen3-4B на kid_adult чтобы он отвечал простым языком.
потом считаем P_simple на public_test_style (seed=42, greedy).

In [ ]:
!pip -q install -U transformers==4.44.2 datasets peft==0.12.0 trl==0.9.6 bitsandbytes accelerate scikit-learn

In [ ]:
import os, random, json
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = "42"

# чтоб трейн был стабильный
from transformers import set_seed
set_seed(SEED)

print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

In [ ]:
# данные могут лежать в разных местах (kaggle input, колаб, локально)
# поэтому перебираю пути и беру тот где реально есть файл
import glob

candidates = [
    "ml-olympiad-red-task",
    "/kaggle/working/ml-olympiad-red-task",
]
# на kaggle датасет лежит в /kaggle/input/<имя>/... - ищем сами
for p in glob.glob("/kaggle/input/**/kid_adult.jsonl", recursive=True):
    candidates.append(os.path.dirname(os.path.dirname(p)))  # это папка где лежит /data

BASE = None
for c in candidates:
    if os.path.exists(c + "/data/kid_adult.jsonl"):
        BASE = c
        break

if BASE is None:
    # папки нет - клоним репу, в ней и данные и метрики
    os.system("git clone https://github.com/Mi5tR/CU_stip_AIRED.git")
    for c in glob.glob("**/kid_adult.jsonl", recursive=True):
        BASE = os.path.dirname(os.path.dirname(c))
        break

print("BASE =", BASE)
DATA = BASE + "/data"
METRICS = BASE + "/metrics"

def read_jsonl(path):
    arr = []
    f = open(path, "r", encoding="utf-8")
    for line in f:
        line = line.strip()
        if line == "":
            continue
        arr.append(json.loads(line))
    f.close()
    return arr

train_data = read_jsonl(DATA + "/kid_adult.jsonl")
test_data = read_jsonl(DATA + "/public_test_style.jsonl")
print(len(train_data), len(test_data))
# print(train_data[0])  # глянуть что внутри

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

In [ ]:
# lora конфиг. r=16 
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# делаю тексты для sft. на входе вопрос, на выходе kid (простой ответ)
from datasets import Dataset

rows = []
for d in train_data:
    q = d["prompt"]
    ans = d["kid"]
    msgs = [
        {"role": "user", "content": q},
        {"role": "assistant", "content": ans},
    ]
    txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    rows.append({"text": txt})

# print(rows[0]["text"])  # проверить формат
ds = Dataset.from_list(rows)
print(len(ds))

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="sft_out",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    bf16=False,
    fp16=True,
    optim="paged_adamw_8bit",
    max_seq_length=768,
    packing=False,
    seed=SEED,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds,
    tokenizer=tok,
)
trainer.train()

In [ ]:
# генерим ответы на public_test_style. greedy, do_sample=False
model.config.use_cache = True
model.eval()

preds = []
for i in range(len(test_data)):
    q = test_data[i]["prompt"]
    msgs = [{"role": "user", "content": q}]
    prompt_ids = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            prompt_ids,
            max_new_tokens=256,
            do_sample=False,
            num_beams=1,
            pad_token_id=tok.pad_token_id,
        )
    gen = out[0][prompt_ids.shape[1]:]
    text = tok.decode(gen, skip_special_tokens=True).strip()
    preds.append(text)
    # if i < 2: print(text)  # проверить пару ответов

print("готово, ответов:", len(preds))
print(preds[0][:200])

In [ ]:
# считаем P_simple через style_clf.pkl
import pickle
import scipy.sparse as sp

d = pickle.load(open(METRICS + "/style_clf.pkl", "rb"))
clf = d["clf"]
v1, v2 = d["vecs"]

X = sp.hstack([v1.transform(preds), v2.transform(preds)])
proba = clf.predict_proba(X)
# класс 1 = простой стиль
idx = list(clf.classes_).index(1)
p_simple = proba[:, idx]

res = float(np.mean(p_simple))
print("P_simple (mean) =", res)